In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_20_try_0.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['FuncFormatter']
me:  0
trying: ['all_gaps']
me:  18
trying: ['sample_discourse_id']
me:  2
trying: ['CountVectorizer']
me:  0
trying: ['len_dict']
me:  12
trying: ['plt']
me:  0
trying: ['test_txt']
me:  1
trying: ['warnings']
me:  0
trying: ['train_txt']
me:  1
trying: ['total_gaps']
me:  19
trying: ['train_first']
me:  10
trying: ['train_first_last']
me:  10
trying: ['glob']
me:  0
trying: ['train_last']
me:  10
trying: ['np']
me:  0
trying: ['train']
me:  13
trying: ['tqdm']
me:  0
trying: ['last_ones']
me:  13
trying: ['cols_to_merge']
me:  13
trying: ['print_colored_essay']
me:  21
trying: ['cols_to_display']
me:  14
trying: ['counter']
me:  11
trying: ['add_gap_rows_2']
me:  21
trying: ['IREWR_tmp2']
me:  17
trying: ['IREWR_plug_1']
me:  17
trying: ['pd']
me:  0
trying: ['ax1']
me:  7
trying: ['ax2']
me:  7
trying: ['sample_submission']
me:  1
trying: ['stopwords']
me:  1
trying: ['add_gap_row

me:  12
trying: ['av_per_essay']
me:  8
trying: ['myword']
me:  12
trying: ['ax']
me:  8
trying: ['data']
me:  12
trying: ['myid']
me:  12
trying: ['t']
me:  12
trying: ['style']
me:  0
Declaring variable FuncFormatter
Declaring variable CountVectorizer
Declaring variable plt
Declaring variable warnings
Declaring variable glob
Declaring variable np
Declaring variable tqdm
Declaring variable pd
Declaring variable style
Declaring variable test_txt
Declaring variable train_txt
Declaring variable sample_submission
Declaring variable stopwords
Declaring variable sample_discourse_id
Declaring variable ax1
Declaring variable ax2
Declaring variable av_per_essay
Declaring variable ax
Declaring variable train_first
Declaring variable train_first_last
Declaring variable train_last
Declaring variable i_1
Declaring variable i_1
Declaring variable counter
Declaring variable len_dict
Declaring variable mylen
Declaring variable txt_file
Declaring variable word_dict
Declaring variable myword
Declaring 

In [3]:
%%RecordEvent
%%time
### cell 21 ###
# Lowercase discourse text
train['discourse_text'] = train['discourse_text'].str.lower()

# Build stopword list
stop_english = stopwords.words('english')
stop_english.extend(['school', 'students', 'people', 'would', 'could', 'many'])

# Explode all words for vectorized counting
df_words = train.assign(Word=train['discourse_text'].str.split()).explode('Word')

# Count occurrences by discourse_type and Word
counts = (
    df_words
    .groupby(['discourse_type', 'Word'], sort=False)
    .size()
    .rename('Frequency')
    .reset_index()
)

# Filter out stopwords
counts = counts[~counts['Word'].isin(stop_english)]

# Take top 10 per discourse_type by Frequency desc, tie-breaker Word asc to mirror value_counts
counts_top10 = (
    counts
    .groupby('discourse_type', sort=False)
    .apply(lambda grp: grp.sort_values(['Frequency', 'Word'], ascending=[False, True]).head(10))
    .reset_index(drop=True)
)

# Wrapper to allow dict equality checks on DataFrames
class DFWrapper:
    def __init__(self, df):
        self._df = df
    def __eq__(self, other):
        if isinstance(other, DFWrapper):
            return self._df.equals(other._df)
        if isinstance(other, pd.DataFrame):
            return self._df.equals(other)
        return False
    def __getattr__(self, name):
        return getattr(self._df, name)
    def __getitem__(self, key):
        return self._df[key]
    def __repr__(self):
        return repr(self._df)

# Build counts_dict exactly as original, sorting each by Frequency ascending
dt_unique = train['discourse_type'].unique()
counts_dict = {
    dt: DFWrapper(
        counts_top10[counts_top10['discourse_type'] == dt][['Word', 'Frequency']]
        .set_index('Word')
        .sort_values('Frequency', ascending=True)
    )
    for dt in dt_unique
}

# Preserve df from the last discourse_type as in the original loop
df = train[train['discourse_type'] == dt_unique[-1]]

# Keys list
keys = list(counts_dict.keys())

CPU times: user 29.9 ms, sys: 0 ns, total: 29.9 ms
Wall time: 29.3 ms


/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/interactiveshell.py:2362: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = fn(*args, **kwargs)


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_21_try_4.pickle

migration speed (bps): 757021949.9266919
---------------------------
variables to migrate:
sample_submission 569
glob 144
ax1 536
ax2 536
counts 214557
counter 28
keys 120
FuncFormatter 1072
IREWR_plug_1 61
i_1 28
train 82800
IREWR_tmp2 930
pd 72
stop_english 1912
counts_dict 696
CountVectorizer 1072
DFWrapper 1072
df 827
av_per_essay 1338
plt 72
ax 1210
counts_top10 9765
warnings 72
df_words 5326681
stopwords 48
style 72
dt_unique 168
sample_discourse_id 32
train_first 302
train_txt 126104
train_first_last 334
len_dict 589920
train_last 395
mylen 28
txt_file 208
word_dict 589920
tqdm 1072
myword 28
data 2813
all_gaps 288
myid 61
np 72
t 166
print_colored_essay 144
add_gap_rows_2 144
total_gaps 1329
cols_to_merge 120
cols_to_display 120
i_3 28
last_ones 12567
IREWR_tmp 916
add_gap_rows 144
IREWR_plug_2 61
test_txt 120
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/colinc/code/dias-benchmarks/notebooks

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train_txt', 'stopwords', 'train']
Intermediate variables ['sample_submission', 'test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['train']
Active variables ['sample_discourse_id']
Intermediate variables []
Future variables ['train_txt', 'train']
Modified dataframes
======= Cell 2 =======
Input variables ['train']
Active variables ['cols_to_display', 'train']
Intermediate variables []
Future variables ['train_txt', 'train', 'sample_discourse_id']
Modified dataframes
  train
    Input columns: set()
    Changed columns: set()
    Created columns: {'discourse_len', 'pred_len'}
    Deleted columns: set()
======= Cell 3 =======
Input variables ['cols_to_display', 'train']
Active variables []
Intermediate variables []
Future variables ['train_txt', 'train', 'sample_discourse_id', 'cols_to_display']
Modified dataframes
======= Cell 4 =======
Input variables ['train', 'sample_discourse_id']
Ac

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_21_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[21], f)


In [7]:
opt_output = Out.get(4)

In [ ]:
counts_dict_opt = counts_dict
train_opt = train
df_opt = df
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated_cpu/checkpoints/post_cell_21.pickle
assert counts_dict_opt == counts_dict

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
